# Блокнот для сбора данных о видео, опубликованных в вк видео
Перед созданием этого блокнота был собран датасет о шоу Medium Quality: канал, шоу, ссылка на плейлист, ведущие, дополнительные комментарии. В блокноте скрейпер делает запрос к API и собирает данные о каждом выпуске. Затем из заголовков и описаний к видео извлекаются имена -- позже они проверялись и корректировались вручную.

In [ ]:
import requests
import pandas as pd

## Чистим данные о шоу, которые собраны вручную

In [ ]:
# читаем датасет, собранный ручную, исправляем мелкие огрехи
albums_df = pd.read_csv('mq.csv', index_col=False)
albums_df.drop(albums_df[albums_df['show_title'].isna()].index, inplace=True)
albums_df['host_norm'] = albums_df['host_norm'].str.strip()
albums_df['host_norm'] = albums_df['host_norm'].replace(' и ', ',', regex=True)
albums_df['host_norm'] = albums_df['host_norm'].replace('\xa0', '', regex=True)
albums_df['host_norm'] = albums_df['host_norm'].replace(', ', ',', regex=True)

In [ ]:
def get_url_params(link):
    """
    Функция извлекает owner_id и album_id из ссылки на плейлист с выпусками
    """
    ownr = []
    alb = []
    for l in link.split(', '):
        # print(l)
        if 'https://vkvideo.ru/playlist/' in l:
            o, a = l.split('/')[-1].split('_')
            ownr.append(o)
            alb.append(a)
        else:
            print('could not process link:', l)
            continue

    if ownr and alb:
        return ','.join(ownr) if len(set(ownr))>1 else ownr[0], ','.join(alb) if len(set(alb))>1 else alb[0]
    return '', ''

albums_df[['owner_id', 'album_id']] = albums_df['link'].apply(get_url_params).to_list()
albums_df

In [ ]:
# создаем столбец с количеством выпусков в каждом плейлисте (=шоу). На старте для всех шоу значение равно 0.
albums_df['videos_count'] = 0
albums_df

In [ ]:
# если шоу поделено на несколько плейлистов, размножить строку, чтобы на одной строке был один плейлист
albums_df_expl = albums_df.copy()
albums_df_expl['link'] = albums_df_expl['link'].str.split(', ')

albums_df_expl = albums_df_expl.explode('link')
albums_df_expl[albums_df_expl['link'].duplicated(keep=False)]

# Достаем данные о видео в каждом плейлисте, отправляя запрос к апи

In [ ]:
import time
import random

def request_data(session, url: str, access_token: str, owner_id: str, album_id: str, count: int=100) -> dict:
    """
    Отправить запрос к API для получения информации по каждому видео, опубликованному в плейлисте по url
    """
    headers = {
        'Origin': 'https://vkvideo.ru',
        # 'Content-Type': 'application/json'
    }

    data = {
        'v': '5.268',
        'client_id': '52461373',
        'album_id': album_id,
        'count': count,
        'extended': 1,
        'fields': 'is_esia_verified%2Cis_sber_verified%2Cis_tinkoff_verified%2Cphoto_50%2Cverified',
        'offset': 0,
        'owner_id': owner_id,
        'sort_album':0,
        'access_token': access_token
    }

    resp = session.post(url=url, data=data, headers=headers)
    text = resp.text
    headers = resp.headers
    return {'resp': resp.json(), 'headers': headers, 'cookies': resp.cookies}

def extract_from_json(idx, owner_id, album_id, json_response):
    """
    Распарсить JSON, вытащить необходимые данные
    """
    res = []

    for i, v in enumerate(json_response['response']['items']):
        # print(v.get('video')['video'])
        if not v['video'].get('title'):
            print('something wrong with', i, v['video'])
            if not failed.get(idx):
                failed[idx] = dict()
            failed[idx][i] = v['video']
            continue
        # print(v['video']['playlist_position'])
        res.append({
            '_album_index': idx,
            '_album_owner_id': owner_id,
            '_album_album_id': album_id,
            'id': v['video'].get('id'),
            'owner_id': v['video'].get('owner_id'),
            'title': v['video'].get('title'),
            'direct_url': v['video'].get('direct_url'),
            'adding_date': v['video'].get('adding_date'),
            'date': v['video'].get('date'),
            'description': v['video'].get('description'),
            'duration': v['video'].get('duration')
        })

    return res

def wait(lower=5, upper=15):
    """
    Выжидаем интервал от LOWER до UPPER секунд
    """
    interval = random.uniform(lower, upper)
    time.sleep(interval)

In [ ]:
access_token = ... # токен

url = 'https://api.vkvideo.ru/method/video.getFromAlbum?v=5.268&client_id=52461373'

failed = {}

def process_links(url, access_token) -> list:
    """
    Запустить сбор данных по видео из плейлиста. Возвращает список со словарями, где один словарь содержит информацию об одном видео
    """
    logs = open('logs.txt', 'w+')
    session = requests.Session()
    data_cleaned = []

    for idx, row in albums_df.iterrows():

        for l in row['link'].split(', '):
            print(idx, l)

            if 'https://vkvideo.ru/playlist/' in l:


                owner_id, album_id = row['owner_id'], row['album_id']

                r = request_data(session, url, access_token, owner_id, album_id)

                albums_df.loc[idx, 'videos_count'] += r['resp']['response']['count']

                if r['resp']['response']['count'] > len(r['resp']['response']['items']):
                    print('ERROR: too many items for', idx, 'link:', l, file=logs)
                    if not failed.get(idx):
                        failed[idx] = dict()
                    failed[idx][l] = 'too many items'
                    continue

                data = r['resp']
                data_cleaned.extend(extract_from_json(idx, owner_id, album_id, data))

                print('OK', idx, 'link:', l, file=logs)

            else:
                print('ERROR: could not process', idx, 'link:', l, file=logs)
                if not failed.get(idx):
                    failed[idx] = dict()
                failed[idx][l] = 'could not process'

            wait()
    logs.close()
    return data_cleaned

data_cleaned = []
data_cleaned.extend(process_links(url, access_token))

In [ ]:
len(data_cleaned)

2885

In [ ]:
# данные об эпизодах
df_episodes = pd.DataFrame.from_dict(data_cleaned)
df_episodes.drop_duplicates(inplace=True)
df_episodes

,_album_index,_album_owner_id,_album_album_id,id,owner_id,title,direct_url,adding_date,date,description,duration
0,0,-5790407,1,456239263,-5790407,УЛИЦА.#16 Лиза Гончарова х Оля Парфенюк,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.702888e+09,1702890000,"Тик-так, время выгоды! Финальная распродажа го...",1739.0
1,0,-5790407,1,456239252,-5790407,УЛИЦА.#15 Мария Орзул х Паша Техник,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.701643e+09,1701680400,DOSTAЕВСКИЙ. Этот корпоратив все запомнят! Заб...,1475.0
2,0,-5790407,1,456239243,-5790407,УЛИЦА.#14 Артём Муратов х Илья Сатир,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.700553e+09,1700557200,Хотите выгодно обновить бытовую технику и элек...,1701.0
3,0,-5790407,1,456239238,-5790407,УЛИЦА.#13 Олеся Иванченко х Дима Журавлёв,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.699261e+09,1699261200,"Ну, короче, Олеся пришла на улицу, супер-класс...",2062.0
4,0,-5790407,1,456239217,-5790407,УЛИЦА.#12 Гоша Карцев х Вика Складчикова,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.696842e+09,1696842130,"Так, ну че, пришел Гоша, мы его развалили, а В...",2523.0
...,...,...,...,...,...,...,...,...,...,...,...
3402,110,-110135406,44,456242137,-110135406,Большое шоу 1 сезон. Вторая серия.,https://vkvideo.ru/playlist/-110135406_44/vide...,1.716914e+09,1676534180,"Это ""Большое шоу"" Азамата Мусагалиева! Десять ...",2051.0
3403,110,-110135406,44,456242141,-110135406,Большое шоу 1 сезон. Третья серия.,https://vkvideo.ru/playlist/-110135406_44/vide...,1.716914e+09,1676616911,"Это ""Большое шоу"" Азамата Мусагалиева! Десять ...",1675.0
3404,110,-110135406,44,456242145,-110135406,Большое шоу 1 сезон. Четвёртая серия.,https://vkvideo.ru/playlist/-110135406_44/vide...,1.716914e+09,1676705211,"Это ""Большое шоу"" Азамата Мусагалиева! Десять ...",1360.0
3405,110,-110135406,44,456242149,-110135406,Большое шоу 1 сезон. Пятая серия.,https://vkvideo.ru/playlist/-110135406_44/vide...,1.716914e+09,1676791605,"Это ""Большое шоу"" Азамата Мусагалиева! Десять ...",2064.0


In [ ]:
df_episodes['link'] = 'https://vkvideo.ru/playlist/' + df_episodes['_album_owner_id'].astype(str) + '_' + df_episodes['_album_album_id'].astype(str)

In [ ]:
# df_episodes.to_csv('mq_episodes.csv', index=False)

In [ ]:
df_episodes['link'].value_counts()

,count
link,
https://vkvideo.ru/playlist/-213802301_1,132
https://vkvideo.ru/playlist/-219105664_5,121
https://vkvideo.ru/playlist/-110135406_44,113
https://vkvideo.ru/playlist/-203677279_4,109
https://vkvideo.ru/playlist/-211229778_1,99
...,...
https://vkvideo.ru/playlist/-217878975_13,2
https://vkvideo.ru/playlist/-109800058_17,2
https://vkvideo.ru/playlist/-214484275_15,1


In [ ]:
albums_df.index = list(albums_df.index)
albums_df.index

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       178, 179, 180, 181, 182, 183, 184, 185, 186, 187],
      dtype='int64', length=188)

In [ ]:
albums_df_expl = albums_df.copy()
albums_df_expl['link'] = albums_df_expl['link'].str.split(', ')

In [ ]:
albums_df_expl = albums_df_expl.explode('link')
albums_df_expl

,label,show_title,release_dates,host,genre,host_norm,link,comment,owner_id,album_id,videos_count
0,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,16
1,Bad Company,KOROТЫШИ,2023 -- наст. время. Ютуб+ВК,"Михаил Стогниенко, Валерий Равдин",NaN,"Михаил Стогниенко,Валерий Равдин",https://vkvideo.ru/playlist/-5790407_2,NaN,-5790407,2,12
2,easy com,Меломан,NaN,Дмитрий Журавлёв,NaN,Дмитрий Журавлев,https://vkvideo.ru/playlist/-109800058_9,NaN,-109800058,9,20
3,easy com,Женский форум,NaN,"Ольга Парфенюк, Варвара Щербакова, Наталья Бор...",разговоры,"Ольга Парфенюк,Варвара Щербакова,Наталья Борис...",https://vkvideo.ru/playlist/-211229778_1,NaN,-211229778,1,99
4,easy com,Утреннее шоу,2023 -- ВК,"Антон Истомин, Евгений Тасенков, Ньургун Атако...",NaN,"Антон Истомин,Евгений Тасенков,Ньургун Атаков,...",https://vkvideo.ru/playlist/-211232966_10,NaN,-211232966,10,47
...,...,...,...,...,...,...,...,...,...,...,...
183,NaN,Внутри Лапенко,NaN,Антон Лапенко,NaN,Антон Лапенко,https://vkvideo.ru/playlist/-220754053_11,NaN,-220754053,11,6
184,NaN,Звёзды,NaN,Азамат Мусагалиев,NaN,Азамат Мусагалиев,https://vkvideo.ru/playlist/-220754053_21,NaN,-220754053,21,15
185,NaN,Кстати,NaN,"Азамат Мусагалиев, Гарик Харламов, Денис Дорохов",NaN,"Азамат Мусагалиев,Гарик Харламов,Денис Дорохов",https://vkvideo.ru/playlist/-220754053_3,NaN,-220754053,3,92
186,NaN,Битва за время,NaN,Саша Ваш.,NaN,Александар Ваш,https://vkvideo.ru/playlist/-220754053_7,NaN,-220754053,7,29


In [ ]:
# денормализуем данные и объединяем информацию о шоу с информацией о выпусках
df = pd.merge(left=albums_df_expl, right=df_episodes, left_on='link', right_on='link', how='left', indicator=True)
df

,label,show_title,release_dates,host,genre,host_norm,link,comment,owner_id_x,album_id,...,_album_album_id,id,owner_id_y,title,direct_url,adding_date,date,description,duration,_merge
0,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,...,1,456239263.0,-5790407.0,УЛИЦА.#16 Лиза Гончарова х Оля Парфенюк,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.702888e+09,1.702890e+09,"Тик-так, время выгоды! Финальная распродажа го...",1739.0,both
1,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,...,1,456239252.0,-5790407.0,УЛИЦА.#15 Мария Орзул х Паша Техник,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.701643e+09,1.701680e+09,DOSTAЕВСКИЙ. Этот корпоратив все запомнят! Заб...,1475.0,both
2,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,...,1,456239243.0,-5790407.0,УЛИЦА.#14 Артём Муратов х Илья Сатир,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.700553e+09,1.700557e+09,Хотите выгодно обновить бытовую технику и элек...,1701.0,both
3,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,...,1,456239238.0,-5790407.0,УЛИЦА.#13 Олеся Иванченко х Дима Журавлёв,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.699261e+09,1.699261e+09,"Ну, короче, Олеся пришла на улицу, супер-класс...",2062.0,both
4,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407,1,...,1,456239217.0,-5790407.0,УЛИЦА.#12 Гоша Карцев х Вика Складчикова,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.696842e+09,1.696842e+09,"Так, ну че, пришел Гоша, мы его развалили, а В...",2523.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3419,NaN,Промпты,NaN,Дмитрий Орлов,NaN,Дмитрий Орлов,https://vkvideo.ru/playlist/-225533288_5,NaN,-225533288,5,...,5,456239249.0,-225533288.0,"Промты: Карина Мейханаджян, Мария Маркова, Але...",https://vkvideo.ru/playlist/-225533288_5/video...,1.747210e+09,1.747210e+09,"Если хочешь разбираться не только в промтах, н...",3418.0,both
3420,NaN,Промпты,NaN,Дмитрий Орлов,NaN,Дмитрий Орлов,https://vkvideo.ru/playlist/-225533288_5,NaN,-225533288,5,...,5,456239227.0,-225533288.0,"Промты: Павел Дедищев, Андрей Бебуришвили, Гур...",https://vkvideo.ru/playlist/-225533288_5/video...,1.745410e+09,1.745410e+09,"Если хочешь разбираться не только в промтах, н...",3298.0,both
3421,NaN,Промпты,NaN,Дмитрий Орлов,NaN,Дмитрий Орлов,https://vkvideo.ru/playlist/-225533288_5,NaN,-225533288,5,...,5,456239218.0,-225533288.0,"Промты: Арсений Попов, Сергей Шевелев, Артем Гаус",https://vkvideo.ru/playlist/-225533288_5/video...,1.744203e+09,1.744203e+09,"Если хочешь разбираться не только в промтах, н...",3687.0,both
3422,NaN,Промпты,NaN,Дмитрий Орлов,NaN,Дмитрий Орлов,https://vkvideo.ru/playlist/-225533288_5,NaN,-225533288,5,...,5,456239185.0,-225533288.0,"Промты: Игорь Джабраилов, Ульяна Пылаева, Алек...",https://vkvideo.ru/playlist/-225533288_5/video...,1.741167e+09,1.741165e+09,Для рекламы и предложений:\nshuuumbrat-svyaz@y...,3777.0,both


In [ ]:
# сохраняем все данные в файлы
for f in ['albums_df', 'albums_df_expl', 'df_episodes', 'df']:
    eval(f).to_csv(f'result/{f}.csv', index=False if f in ('albums_df', 'df') else True)
    eval(f).to_excel(f'result/{f}.xlsx', index=False if f in ('albums_df', 'df') else True)

In [ ]:
import pickle

# сохраняем список со словарями в файл
with open('result/scraped.pickle', 'wb') as handle:
    pickle.dump(data_cleaned, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open('filename.pickle', 'rb') as handle:
#     b = pickle.load(handle)

# print(a == b)

In [ ]:
with open('result/scraped.pickle', 'rb') as handle:
    b = pickle.load(handle)

print(data_cleaned == b)

True


In [ ]:
len(b)

3407

In [ ]:
!zip -r /content/mq.zip /content/result

  adding: content/result/ (stored 0%)
  adding: content/result/albums_df.csv (deflated 79%)
  adding: content/result/albums_df_expl.csv (deflated 78%)
  adding: content/result/scraped.pickle (deflated 81%)
  adding: content/result/albums_df.xlsx (deflated 3%)
  adding: content/result/df.csv (deflated 83%)
  adding: content/result/df_episodes.csv (deflated 81%)
  adding: content/result/df.xlsx (deflated 2%)
  adding: content/result/df_episodes.xlsx (deflated 0%)
  adding: content/result/albums_df_expl.xlsx (deflated 3%)


## Извлекаем гостей с помощью spacy

In [ ]:
import re
import pandas as pd

In [ ]:
df = pd.read_csv('df.csv')
df.head(50)

,label,show_title,release_dates,host,genre,host_norm,link,comment,owner_id_x,album_id,...,_album_album_id,id,owner_id_y,title,direct_url,adding_date,date,description,duration,_merge
0,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239263.0,-5790407.0,УЛИЦА.#16 Лиза Гончарова х Оля Парфенюк,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.702888e+09,1.702890e+09,"Тик-так, время выгоды! Финальная распродажа го...",1739.0,both
1,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239252.0,-5790407.0,УЛИЦА.#15 Мария Орзул х Паша Техник,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.701643e+09,1.701680e+09,DOSTAЕВСКИЙ. Этот корпоратив все запомнят! Заб...,1475.0,both
2,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239243.0,-5790407.0,УЛИЦА.#14 Артём Муратов х Илья Сатир,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.700553e+09,1.700557e+09,Хотите выгодно обновить бытовую технику и элек...,1701.0,both
3,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239238.0,-5790407.0,УЛИЦА.#13 Олеся Иванченко х Дима Журавлёв,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.699261e+09,1.699261e+09,"Ну, короче, Олеся пришла на улицу, супер-класс...",2062.0,both
4,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239217.0,-5790407.0,УЛИЦА.#12 Гоша Карцев х Вика Складчикова,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.696842e+09,1.696842e+09,"Так, ну че, пришел Гоша, мы его развалили, а В...",2523.0,both
5,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239196.0,-5790407.0,УЛИЦА.#11 Наталья Краснова х Саша Ваш,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.694417e+09,1.694423e+09,На Улицу пришла Наташа Краснова и было даже же...,1813.0,both
6,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239178.0,-5790407.0,УЛИЦА.#10 Евгения Медведева x Александр Емелья...,https://vksport.vkvideo.ru/playlist/-5790407_1...,1.692003e+09,1.691999e+09,На Улицу пришла Евгения и пацаны вообще очень ...,1382.0,both
7,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239161.0,-5790407.0,УЛИЦА.#9 Вячеслав Чепурченко х Яна Кошкина,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.689582e+09,1.689580e+09,На Улицу залетел Слава и попросил накидать ему...,2518.0,both
8,Bad Company,Улица,2023 -- наст. время. Ютуб+ВК,"Арсен Аветисян, Ярослав Дудин, Михаил Стогниен...",NaN,"Арсен Аветисян,Ярослав Дудин,Михаил Стогниенко...",https://vkvideo.ru/playlist/-5790407_1,NaN,-5790407.0,1,...,1.0,456239146.0,-5790407.0,УЛИЦА.#8 Надежда Сысоева x Эльдар Джарахов,https://vkvideo.ru/playlist/-5790407_1/video-5...,1.687165e+09,1.687157e+09,Этот выпуск рекомендуется к просмотру под буты...,1734.0,both
9,Ba

In [ ]:
import spacy
nlp = spacy.load("ru_core_news_sm")

def extract_names(text):
    if text:
        doc = nlp(text)
        res = []
        for e in doc.ents:
            if e.label_ == 'PER':
                res.append(e.text)
        return res

guests_from_description = df['description'].fillna('').apply(extract_names)
guests_from_description

,description
0,"[Лиза Гончарова, Оля Парфенюк]"
1,"[Мария, Паша Техник]"
2,"[СберСпасибо, Мишанджелес, Артема Муратова, Ил..."
3,"[Олеся, Дима, Кайф]"
4,"[Гоша, Вика]"
...,...
3419,"[Карина Мейханаджян, Мария Маркова, Александра..."
3420,"[Дима Орлов, Димы Орлова, Павел Дедищев, Андре..."
3421,"[Дима Орлов, Златой, Сергей Шевелев, Арсений П..."
3422,"[Выпуски, Дима Орлов, Игорь Джабраилов, Ульяна..."


In [ ]:
guests_from_title = df['title'].fillna('').apply(extract_names)
guests_from_title

,title
0,"[Лиза Гончарова, Оля Парфенюк]"
1,"[Мария Орзул, Паша Техник]"
2,"[Артём Муратов, Илья Сатир]"
3,"[Олеся Иванченко, Дима Журавлёв]"
4,"[Гоша Карцев, Вика Складчикова]"
...,...
3419,"[Карина Мейханаджян, Мария Маркова, Александра..."
3420,"[Павел Дедищев, Андрей Бебуришвили, Гурам Амарян]"
3421,"[Арсений Попов, Сергей Шевелев, Артем Гаус]"
3422,"[Игорь Джабраилов, Ульяна Пылаева, Алексей Шал..."


In [ ]:
df['guests_from_title'] = guests_from_title
df['guests_from_description'] = guests_from_description

In [ ]:
d = dict()

for item in list(guests_from_title + guests_from_description):
    if type(item)==list:
        for name in item:
            if ' ' in name:
                print(name)
                try:
                    first_name, last_name = name.split()[:2]
                    d[last_name] = name
                except:
                    pass


In [ ]:
df.to_csv('df.csv', index=False)
df.to_excel('df.xlsx', index=False)